# 散度加权优势函数（Divergence-Weighted Advantage）


## 动机：为什么要给关键 Token 更高权重？
###  论文的核心发现回顾
在提出散度加权优势之前，论文已经通过大量实验建立了以下关键事实：

| 发现        | 含义                                  |
| --------- | ----------------------------------- |
| **稀疏性**   | 83%~98% 的 token 位置分布几乎不变            |
| **关键性**   | <font color='red'>少量高散度 token 决定了序列级推理性能</font>              |
| <font color='red'>**重排序机制** </font>| RLVR 主要是在 base 模型的候选集中"选择"更好的 token |

#### 核心洞察：
- <font color='blue'>既然只有极少数 token 真正驱动了性能提升，那么均匀对待所有 token 的 RLVR 训练可能是计算资源的浪费。</font>
###  现有 RLVR 的问题
<font color='red'>标准 RLVR（如 GRPO/DAPO）的优势信号对所有 token 一视同仁：</font>

$A_t$=group-relative advantage 

这意味着：
- <font color='red'>低散度 token（已足够好）获得与关键 token 相同的梯度更新强度</font>
- <font color='blue'>计算资源被均匀分散在整个序列上</font>
- <font color='red'>可能产生不必要的分布偏移（即使这些 token 不需要改变）</font>

# 散度加权优势函数的设计
##  核心思想
- <font color='red'> 利用 token 级 JS 散度作为"重要性权重"</font>，对高散度（关键）token 给予更高的优势信号，对低散度 token 给予更低权重。

##  数学定义
论文提出将标准优势信号$A_t$乘以基于 JS 散度的权重函数：

$\hat{A}{~}_t = w(JS_t)⋅A_t $

### 其中：

-  $JS_t = D_{JS}(π_{base}(⋅∣x <t) || π_{RL}(⋅∣x <t))$ 是位置 t  的 Jensen-Shannon 散度
- w(⋅)  是权重函数，<font color='red'>将散度映射到优势缩放因子</font>

##  <font color='red'>权重函数的选择</font>


| 权重函数类型         | 公式                                   | 特性           |
| -------------- | ------------------------------------ | ------------ |
| **线性加权**       | $w(JS) = 1 + \alpha \cdot JS$        | 简单，单调递增      |
| **阈值加权**       | $w(JS) = \mathbb{1}[JS > \tau]$      | 只关注高散度 token |
| **Softmax 加权** | $w(JS) \propto \exp(\beta \cdot JS)$ | 指数强调极端值      |
| **归一化加权**      | $w(JS) = JS / \max(JS)$              | 相对重要性        |


# 为什么这种方法有效？
##  理论直觉
###  信号聚焦 (Signal Focusing)
标准 RLVR 的梯度更新：$∇_θJ = E[\sum_tA_t ⋅ ∇_θlogπ_(a_t|s_t)]$

散度加权后的更新：$∇_θJ_{weighted} = E[\sum_tw(JS_t) ⋅ A_t ∇_θlogπ_(a_t|s_t)]$

- 高散度 token → <font color='red'>$w(JS_t)$ </font> 大 → 更强的梯度信号 → 加速关键决策的学习
- 低散度 token → <font color='red'>$w(JS_t)$ </font> 小 → 抑制不必要的更新 → 减少分布偏移

###  与 Cross-Sampling 实验的呼应
论文的 Cross-Sampling 实验证明：
- <font color='red'>替换少量高散度 token 即可恢复 RL 性能</font>
  - 这直接支持了"高散度 token 值得更多关注"的假设。
###  <font color='red'>计算效率</font>
- <font color='blue'>不需要为低散度 token 计算完整的梯度更新</font>
- <font color='red'>可以设计早停机制：当 $JS_t<ϵ$  时跳过该 token 的梯度计算</font>
- **训练速度提升与稀疏性成正比（83%~98% 的 token 可被跳过）**

# 实验结果与发现
##  性能改进
论文报告了散度加权变体在多个设置下的结果：

| 设置               | 基线 | 散度加权 | 改进 |
| ---------------- | -- | ---- | -- |
| Qwen2.5-32B DAPO | X% | X+Δ% | 正向 |
| Qwen2.5-Math-7B  | Y% | Y+Δ% | 正向 |

### <font color='red'>关键发现：</font>
- 散度加权可以超越标准基线
- 但最优权重函数是模型特定的（需要调参）

##  训练动态分析
图（论文中的相关图表）展示了：
- 梯度幅度分布
  - 标准 RLVR：梯度在整个序列上相对均匀
  - 散度加权：<font color='red'>梯度高度集中在高散度位置</font>
- JS 散度演化
  - 加权训练下，高散度 token 的散度增长更快
  - 低散度 token 的散度被抑制
- 收敛速度
  - 在部分设置中，加权版本收敛更快

# 实现细节与工程考量
##  计算开销

| 组件      | 标准 RLVR  | 散度加权 RLVR             |
| ------- | -------- | --------------------- |
| JS 散度计算 | 无        | <font color='red'>需要（但可在 top-p 截断分布上近似）</font> |
| 优势计算    | $O(T)$   | $O(T)$ + 权重乘法         |
| 梯度更新    | 所有 token | 可跳过低权重 token          |
| **总开销** | 基准       | **可能更低**（如果实现稀疏梯度）    |

## 实际实现策略

In [ ]:
# 伪代码示意
def divergence_weighted_advantage(logits_base, logits_rl, advantage, top_p=0.9):
    """
    logits_base: base model 的 logits
    logits_rl: RL model 的 logits  
    advantage: 标准 group-relative advantage
    """
    # 1. 计算 top-p 截断分布
    dist_base = top_p_truncated_softmax(logits_base, top_p)
    dist_rl = top_p_truncated_softmax(logits_rl, top_p)
    
    # 2. 计算 JS 散度
    js_divergence = compute_js_divergence(dist_base, dist_rl)
    
    # 3. 应用权重函数
    weights = weight_function(js_divergence)  # 如: 1 + alpha * js
    
    # 4. 加权优势
    weighted_advantage = weights * advantage
    
    # 5. 可选：稀疏化——只更新高权重 token
    mask = weights > threshold
    weighted_advantage = weighted_advantage * mask
    
    return weighted_advantage

## 超参数调优
<font color='red'>论文指出权重函数的选择影响显著：</font>


| 超参数                  | 作用                   | 调优建议        |
| -------------------- | -------------------- | ----------- |
| $\alpha$ (线性系数)      | <font color='red'>控制加权强度  </font>             | 小值开始，逐步增大   |
| $\tau$ (<font color='red'>阈值</font>)          | 决定"关键 token"的 cutoff | 基于散度分布的百分位  |
| $\beta$ (Softmax 温度) | <font color='red'>控制聚焦程度  </font>             | 高值 = 更极端的聚焦 |


# 局限性与未来方向
## <font color='red'>论文指出的局限</font>
1. 模型特异性
   - "stable performance gains may require model-specific selection of the weighting function"
   - <font color='red'>不同模型、不同任务可能需要不同的权重函数。</font>
2. 与 clip-higher 的交互
   - **<font color='blue'>DAPO 的 clip-higher 机制已经改变了探索行为，与散度加权的交互需要仔细研究。</font>**
3. 长程依赖
   - <font color='red'>当前分析主要关注局部 token 散度，全局轨迹结构的影响尚未完全捕捉。</font>
  
## 潜在改进方向

| 方向             | 想法                 |
| -------------- | ------------------ |
| **动态权重**       | 训练过程中自适应调整权重函数     |
| **结构感知**       | 结合位置信息（开头/结尾的特殊处理） |
| **任务自适应**      | 根据任务类型自动选择最优权重     |
| **与 KL 正则化结合** | 用散度加权替代或补充 KL 约束   |

# 与其他工作的联系
## 与 "RL's Razor" 的互补

| 论文                                     | 视角             | 散度加权的角色             |
| -------------------------------------- | -------------- | ------------------- |
| **RL's Razor** (Shenfeld et al., 2025) | 序列级 KL 最小化解释遗忘 | 在 token 级实现"最小必要更新" |
| **Sparse but Critical** (本文)           | Token 级稀疏性     | 利用稀疏性设计更高效的训练目标     |

### 统一视角： 
- <font color='red'>两篇论文都表明 RL 的成功在于保守的、有针对性的更新。散度加权优势是将这一洞察操作化为训练算法。</font>

##  <font color='blue'>与课程学习的类比</font>
散度加权可以看作一种"token 级课程学习"：
- 先学习最重要的决策（高散度 token）
- 逐步减少对已掌握内容的关注（低散度 token）

# 一句话总结
- 散度加权优势函数的核心洞察是：
- 既然 RLVR 的成功只依赖于极少数关键 token，那么训练时就应该把计算资源和梯度信号集中投放到这些 token 上——这不仅可能提升性能，还能让训练更高效、更聚焦。